In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章在一维二次函数 + 1000 个 2D 点上演示优化器与调度，CPU 完全够用。
# 之所以仍要打印 CUDA 信息，是为了和后续 GPU 章节保持同一份"开机自检"模板，
# 读者切到任何一章都能第一眼确认运行时是否如预期。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())   # bool；False 也无所谓，本章不需要 GPU
if torch.cuda.is_available():
    # GPU 名字常见取值：'Tesla T4' / 'NVIDIA L4' / 'NVIDIA A100-SXM4-40GB'
    print('Device         :', torch.cuda.get_device_name(0))
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')


In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行，否则会被当 line magic 报错。
# scikit-learn: make_moons 玩具数据
# matplotlib:   画优化轨迹、lr 曲线、loss 曲线
# torch 在 Colab 已预装。
!pip install -q -U scikit-learn matplotlib


In [ ]:
# ============================================================
# Cell 2: SGD vs Momentum vs Adam —— 在一个二次函数上画轨迹
# ============================================================
# 目标函数：f(x, y) = 0.5 * (x^2 + 10 * y^2) —— 长椭圆山谷
# 不同方向曲率差 10 倍，纯 SGD 会沿 y 方向震荡。
# 系数 0.5 是"传统写法"——求导后正好把 2 消掉，∂f/∂x = x、∂f/∂y = 10y，
# 公式更干净；不写 0.5 也不影响结论，只是梯度数值会翻倍。
import torch
import matplotlib.pyplot as plt

def loss_fn(p):
    # p 是形状 (2,) 的张量；p[0] 当 x、p[1] 当 y
    # 返回标量 tensor，能直接 .backward()
    return 0.5 * (p[0] ** 2 + 10 * p[1] ** 2)

def run(optimizer_cls, kwargs, lr, n_steps=40, init=(-3.0, 1.5)):
    # init=(-3, 1.5) 离最优点 (0,0) 远、且在 y 方向有偏置——能把"y 方向震荡"看清楚
    # requires_grad=True 是关键：让这个 tensor 进入 autograd 图，反向传播时才会被填 grad
    p = torch.tensor(init, requires_grad=True)
    # optimizer 接收一个"参数列表"——这里只有 p 一个 tensor
    # **kwargs 里塞 momentum / betas 等优化器特有超参
    opt = optimizer_cls([p], lr=lr, **kwargs)
    # detach() 切断 autograd（轨迹只用来画图、不要再求导）；clone() 拷一份避免后续 in-place 修改影响历史；tolist() 转成纯 Python 方便 numpy 收集
    traj = [p.detach().clone().tolist()]
    for _ in range(n_steps):
        loss = loss_fn(p)
        opt.zero_grad()    # 清掉上一轮残留梯度，否则会累加
        loss.backward()    # 反向传播，把 dL/dp 写入 p.grad
        opt.step()         # 优化器读 p.grad、按各自规则更新 p
        traj.append(p.detach().clone().tolist())
    return traj

# 三种优化器跑同样的初始点、同样步数，唯一区别是更新规则。
# 三个 lr 都是手调过的，各自取"能凸显该优化器特征"的值：
#   SGD lr=0.18：把 y 方向更新因子 (1 - lr·10) 推到 -0.8——既负又 |·|<1 →
#                每步在 y 方向跨过 0 跳到对面、振幅按 0.8 衰减，画面上是密集的 ping-pong；
#   Momentum lr=0.05：故意保持小 lr——动量把"每步翻一次"压成"每几步才翻一次"的慢波；
#   Adam lr=0.3：内部 √v̂ 把梯度量级归一了，要用大得多的 lr 才能看出明显推进。
trajs = {
    'SGD lr=0.18':           run(torch.optim.SGD,  {},                              lr=0.18),
    'SGD+Momentum lr=0.05':  run(torch.optim.SGD,  {'momentum': 0.9},               lr=0.05),
    'Adam lr=0.3':           run(torch.optim.Adam, {'betas': (0.9, 0.999)},         lr=0.3),
}

# 画等高线 + 三条优化轨迹
import numpy as np
# 把 x、y 平面切成 200×200 网格，准备算每个网格点的 loss 值用来画等高线
# x 范围右探到 +1.5：Adam 在 lr=0.3 下会沿对角线越过原点冲到 x>0 一侧再绕回来，太窄会把这段轨迹裁掉
xs = np.linspace(-3.5, 1.5, 200)
ys = np.linspace(-2.0, 2.0, 200)
XX, YY = np.meshgrid(xs, ys)            # XX、YY 形状都是 (200, 200)
ZZ = 0.5 * (XX ** 2 + 10 * YY ** 2)     # 同样形状，每格存对应 (x, y) 的 loss

fig, axes = plt.subplots(1, 3, figsize=(13, 4))   # 3 子图横排
for ax, (name, traj) in zip(axes, trajs.items()):
    # 灰色等高线：levels=20 表示画 20 条等高线；alpha=0.5 让轨迹更突出
    ax.contour(XX, YY, ZZ, levels=20, cmap='Greys', alpha=0.5)
    traj = np.array(traj)                                    # 形状 (n_steps+1, 2)
    ax.plot(traj[:, 0], traj[:, 1], '-o', markersize=3)      # 轨迹点 + 连线
    ax.scatter([0], [0], marker='*', color='red', s=100, label='optimum')
    ax.set_title(name)
    # aspect='equal' 让 x、y 单位长度一致，否则椭圆山谷会被画成圆，看不出曲率差异
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print('观察：')
print('  纯 SGD（lr=0.18）—— y 方向更新因子 = 1-0.18·10 = -0.8，')
print('                       每一步都跨过 0 跳到对面壁，画面里能直接数出密集的 ping-pong；')
print('                       x 方向则一路单调缩到 0')
print('  +Momentum（lr=0.05）—— 也仍会过零，但频率明显降低（每 ~3 步才翻一次而非每步），')
print('                          整段像一条缓慢起伏的波，振荡幅度也比 SGD 小')
print('  Adam（lr=0.3）—— 自适应步长把 x、y 两方向的更新量都归一到 ≈ ±lr，前几步走 45° 对角')


In [ ]:
# ============================================================
# Cell 2b: 动量的手算数值例子 —— 沿谷加速 vs 垂直衰减
# ============================================================
# 设 θ ∈ R^2，连续若干步梯度：
#   g = (0.1, +1.0)   x 分量同号（沿谷方向），y 分量正负交替（垂直方向）
#   g = (0.1, -1.0)
#   g = (0.1, +1.0)
#   g = (0.1, -1.0)
# 动量系数 μ=0.9，初始 v_0 = (0, 0)，按 v_t = μ v_{t-1} + g_t 逐步算。
mu = 0.9
grads = [(0.1, +1.0), (0.1, -1.0), (0.1, +1.0), (0.1, -1.0)] * 4   # 跑 16 步看稳态

v = (0.0, 0.0)
print(f"{'t':>3} {'g_t':>18} {'v_t':>22}")
for t, (gx, gy) in enumerate(grads, start=1):
    v = (mu * v[0] + gx, mu * v[1] + gy)
    print(f'{t:>3}  ({gx:+.3f}, {gy:+.3f})   ({v[0]:+.4f}, {v[1]:+.4f})')

# 解析稳态：
#   x 方向梯度恒为 g_x=0.1，v_x → g_x / (1-μ) = 1.0
#   y 方向 ±1 交替，解 v_+ = μ v_- + 1 与 v_- = μ v_+ - 1
#                     ⇒ v_± = ± 1 / (1+μ) = ±1/1.9 ≈ ±0.526
v_x_inf = 0.1 / (1 - mu)
v_y_peak_inf = 1.0 / (1 + mu)
print()
print(f'稳态 v_x ≈ {v_x_inf:.3f}        (是单步梯度 0.1 的 {v_x_inf / 0.1:.0f} 倍)')
print(f'稳态 |v_y| ≈ {v_y_peak_inf:.3f}     (单步梯度 1.0 衰减到 {v_y_peak_inf:.0%})')
print()
print('对照纯 SGD：每步 update 直接是 g_t 本身，')
print('  沿谷方向 |update| = 0.1，垂直方向 |update| = 1.0')
print('  → 更新被垂直分量主导（1:10），所以 y 方向反复横跳。')
print('Momentum 把 v_x 抬到 1.0、|v_y| 压到 0.526，')
print('  → 两方向之比从 0.1:1.0 反转为 1.0:0.526，主导方向变成沿谷方向。')


In [ ]:
# ============================================================
# Cell 2c: Adam 自适应步长的手算数值例子
# ============================================================
# 设两个参数 a 和 b，每步梯度都恒定：
#   g_a = 1.0
#   g_b = 0.001    (与 g_a 差 1000 倍)
# 默认超参 β1 = 0.9, β2 = 0.999, η = 1e-3, ε ≈ 0；初始 m_0 = v_0 = 0。
# 看 Adam 怎么把 1000 倍梯度差异在 update 量级上抹平。
import math

b1, b2, eta, eps = 0.9, 0.999, 1e-3, 0.0
n_steps = 5

print(f"{'t':>3} | {'name':>3} {'g':>10} {'m':>14} {'v':>14} {'m_hat':>14} {'v_hat':>14} {'update':>14} {'|update|/η':>10}")
print('-' * 110)

for label, g in [('a', 1.0), ('b', 1e-3)]:
    m, v = 0.0, 0.0
    for t in range(1, n_steps + 1):
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        m_hat = m / (1 - b1 ** t)
        v_hat = v / (1 - b2 ** t)
        upd = -eta * m_hat / (math.sqrt(v_hat) + eps)
        print(f'{t:>3} | {label:>3} {g:>10.6g} {m:>14.6e} {v:>14.6e} {m_hat:>14.6e} {v_hat:>14.6e} {upd:>14.6e} {abs(upd)/eta:>10.4f}')
    print()

print('要点：')
print('  1) 偏差修正每一步把 m_hat → g、v_hat → g^2（恒定梯度下精确成立）')
print('  2) 参数 a 和 b 的 g 差 1000 倍、m_hat 差 1000 倍、√v_hat 也差 1000 倍')
print('     → 上下相除后绝对量级被消掉，update 都是 −η = −1e-3，1000:1 的差异被抹平')
print()
print('对照：跳过偏差修正会怎样？（直接用 m / √v）')
for label, g in [('a', 1.0), ('b', 1e-3)]:
    # 只看 t=1
    m1 = (1 - b1) * g           # = 0.1 * g
    v1 = (1 - b2) * g * g       # = 1e-3 * g^2
    upd_unc = -eta * m1 / math.sqrt(v1)
    print(f'  t=1, param {label}: 未修正 update = {upd_unc:.4e}, |update|/η = {abs(upd_unc)/eta:.4f}')
print('  → 两边仍齐平（1000:1 的差异照样被消），但绝对量级被多放大 ~3.16 倍。')
print('  → 这就是 §4.2 偏差修正不能省的原因：零起步会让开头几百步的 update 偏大。')


In [ ]:
# ============================================================
# Cell 3: 手写一遍 Adam，验证与 PyTorch 内置一致
# ============================================================
# Adam 的更新公式（含 bias correction）：
#   m ← β1 m + (1-β1) g
#   v ← β2 v + (1-β2) g^2
#   m̂ = m / (1 - β1^t),  v̂ = v / (1 - β2^t)
#   θ ← θ - lr · m̂ / (√v̂ + eps)
# 思路：对同一个初值 + 同一组"梯度序列"，分别让 PyTorch 内置 Adam 和手写 Adam 各跑一遍，
# 最后用 torch.allclose 比对——能对上才说明上面四行公式没看错。
import torch

torch.manual_seed(0)
# 共享初值 + 同样的"梯度序列"，看两种实现是否一致
def make_grads():
    # 直接用 randn 模拟梯度，不真的去做 forward/backward——
    # 这样测试"优化器本身"的正确性，与模型/loss 解耦
    return [torch.randn(3) for _ in range(20)]   # 20 步，每步形状 (3,)

grads = make_grads()
# 超参取 PyTorch Adam 默认值，方便和内置对齐
lr, b1, b2, eps = 1e-2, 0.9, 0.999, 1e-8

# (a) PyTorch 内置 Adam
theta_a = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([theta_a], lr=lr, betas=(b1, b2), eps=eps)
for g in grads:
    opt.zero_grad()
    # 通常 .grad 是 backward() 写进去的；这里测试优化器本身，所以手动塞预先采好的 g
    # .clone() 防止优化器内部对 grad 做 in-place 操作改坏 grads 列表
    theta_a.grad = g.clone()                       # 直接灌入梯度，绕过 backward
    opt.step()

# (b) 手写 Adam
# 注意 theta_b 不开 requires_grad——我们自己手动算 update，不走 autograd
theta_b = torch.zeros(3)
m = torch.zeros(3); v = torch.zeros(3)             # 一阶矩 / 二阶矩，零初始化（与 PyTorch 一致）
for t, g in enumerate(grads, start=1):             # t 从 1 开始（公式里 β^t 的 t）
    m = b1 * m + (1 - b1) * g                      # 一阶矩：梯度的指数移动平均
    v = b2 * v + (1 - b2) * g * g                  # 二阶矩：梯度平方的指数移动平均（逐元素）
    m_hat = m / (1 - b1 ** t)                      # 偏差修正——抵消零起步导致的早期偏低
    v_hat = v / (1 - b2 ** t)
    # 这里写成 "theta_b = theta_b - ..." 而不是 in-place，是因为 theta_b 不在 autograd 图里、不需要 in-place 优化
    theta_b = theta_b - lr * m_hat / (v_hat.sqrt() + eps)

print('PyTorch Adam   :', theta_a.detach().tolist())
print('手写  Adam     :', theta_b.tolist())
# atol=1e-6 留足浮点误差余地——两边一致才通过
print('两种实现一致 :', torch.allclose(theta_a.detach(), theta_b, atol=1e-6))


In [ ]:
# ============================================================
# Cell 4: lr 太大 / 合适 / 太小 —— 同一个模型同一份数据三种结果
# ============================================================
# 用 P02 的 make_moons 做演示，模型固定为 2 → 16 → 1 的 ReLU MLP。
# 控制变量法：模型结构、初始化（同一 seed）、数据完全一致，唯一变的就是 lr，
# 这样 loss 曲线的差异只能归因于 lr 选择。
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# make_moons: 两个互相缠绕的半月形二分类数据；noise=0.2 加点高斯噪声让它非线性可分但不平凡
# 1000 个样本一次性当 full-batch 训练（数据小，没必要分 mini-batch）
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)         # 形状 (1000, 2)
y = torch.tensor(y_np, dtype=torch.float32)         # 形状 (1000,)，取值 {0, 1}

def train_with_lr(lr, n_epoch=200, seed=0):
    # 同一个 seed 保证每次模型初始化一模一样——只有 lr 在变
    torch.manual_seed(seed)
    # 隐层宽 16 已足够拟合两个半月；最后一层输出 1 维 logit（不过 sigmoid，后面用 "_with_logits" loss）
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    losses = []
    for _ in range(n_epoch):
        # model(X) 形状 (1000, 1)；squeeze(-1) 把最后一维压掉变 (1000,) 与 y 形状对齐
        logits = model(X).squeeze(-1)
        # _with_logits 版把 sigmoid + BCE 数值稳定地合在一起算——不要先手动 sigmoid 再算 BCE
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())             # .item() 把标量 tensor 转成 Python float
    return losses

# 三种 lr：太大会震荡甚至爆炸，太小慢得离谱
# 选值的尺度提示：BCE 默认 reduction='mean'，已经把 1000 个样本的梯度平均，
# 所以"看起来够大"的 lr 实际只相当于 sum-reduction 下的 lr/1000——
# 想真的踩到不稳定区，lr 大概要拉到两位数；30 就够把 loss 反复甩出稳定区，
# 又不至于像 lr=500 那样一冲就被 ylim 截顶、看不出震荡形状。
# 0.001 故意小一两个量级，看"明明在下降但永远到不了"
lrs = [30.0, 0.1, 0.001]
plt.figure(figsize=(7, 4))
for lr in lrs:
    losses = train_with_lr(lr)
    plt.plot(losses, label=f'lr = {lr}')
plt.xlabel('epoch'); plt.ylabel('loss')
# 大 lr 的曲线偶尔会冲到几十，不限上限会把 lr=0.1/0.001 两条压成贴底的细线
plt.ylim(0, 5)
plt.title('Same model, only lr varies: lr is the most sensitive hyperparameter')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('观察：')
print('  lr=30   —— loss 大幅震荡、撞过山谷反复横跳；步子彻底超出稳定区')
print('  lr=0.1  —— 平稳收敛到低位；这是合适的范围')
print('  lr=0.001 —— 下降但慢得离谱；同样的 epoch 远未收敛')


In [ ]:
# ============================================================
# Cell 5: 把 warmup + cosine 调度的 lr 曲线画出来
# ============================================================
# 写法上 PyTorch 里有两种：
#   (a) 用 LambdaLR 自己写 lr_lambda（最直观，便于理解）
#   (b) 用 transformers 的 get_cosine_schedule_with_warmup（生产实操）
# 这里展示 (a)，把"调度本质上就是把 step 映射成 lr 倍率"看透。
# 关键点：lr_lambda 返回的是"乘数"——optimizer 里设的 lr 才是基准，调度器给它乘一个 0~1 的系数。
import math
import torch
import matplotlib.pyplot as plt

peak_lr = 1.0       # 用 1.0 方便看比例（实际训练里这里通常是 1e-4 ~ 6e-4 量级）
T_w = 200           # warmup step 数（前 200 步把 lr 从 0 升到 peak）
T   = 2000          # 总 step 数（warmup + cosine 全程）
eta_min_ratio = 0.1  # cosine 退火下界 = 10% peak（不到 0，避免最后几乎不更新）

def lr_lambda(step):
    # 返回 lr 相对 peak 的"倍率"——LambdaLR 会把它乘到 optimizer 设的基准 lr 上
    if step < T_w:
        # 线性 warmup：0 → 1
        # step=0 → 0；step=T_w-1 → (T_w-1)/T_w ≈ 1；step=T_w 时进入下面的 cosine 段
        return step / T_w
    # cosine decay：1 → eta_min_ratio
    # progress 是"进入 cosine 段后的归一化进度"：T_w 时 = 0、T 时 = 1
    progress = (step - T_w) / (T - T_w)
    # 拆开看：cos(0)=1 → 整体 = eta_min_ratio + (1-eta_min_ratio) = 1
    #        cos(π)=-1 → 整体 = eta_min_ratio + 0 = eta_min_ratio
    # 即 progress 从 0 到 1 时，lr 倍率从 1 平滑滑到 eta_min_ratio
    return eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + math.cos(math.pi * progress))

steps = list(range(T))
lrs = [peak_lr * lr_lambda(s) for s in steps]

plt.figure(figsize=(8, 3.5))
plt.plot(steps, lrs)
# 竖虚线标 warmup 末尾，让"线性段→cosine 段"的转折点一目了然
plt.axvline(T_w, color='gray', linestyle='--', label=f'warmup end (step={T_w})')
plt.axhline(peak_lr * eta_min_ratio, color='green', linestyle=':', label=f'η_min = {eta_min_ratio:.0%} peak')
plt.xlabel('step'); plt.ylabel('lr')
plt.title('warmup + cosine: default LR schedule shape for LLM training')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# 真实训练里就把 lr_lambda 喂给 LambdaLR：
# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# 训练循环里  optimizer.step()  之后调用  scheduler.step()
print('注意：scheduler.step() 必须在 optimizer.step() 之后调，否则 lr 会推迟一步生效。')


In [ ]:
# ============================================================
# Cell 6: 综合训练对比 —— SGD / SGD+Momentum / Adam / AdamW
# ============================================================
# 把 P02 的 make_moons 二分类延续过来，比四种优化器的 loss 曲线。
# 模型、初始化、数据完全一致；区别只在优化器与 lr。
# 注意 lr 是为每个优化器单独"调到合理"的——直接把所有优化器都用同一个 lr 比并不公平
# （Adam 系自适应步长后等效 lr 与 SGD 不在一个量级）。
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

def make_model(seed=42):
    # 固定 seed=42 保证四种优化器拿到完全一样的初始权重
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

def train(opt_cls, opt_kwargs, n_epoch=200):
    model = make_model()                                # 每次重新建模型，从同一初始化起步
    opt = opt_cls(model.parameters(), **opt_kwargs)     # 用 ** 把 lr / momentum / weight_decay 等传进去
    losses = []
    for _ in range(n_epoch):
        logits = model(X).squeeze(-1)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()    # 训练三件套写一行省点篇幅
        losses.append(loss.item())
    return losses

# 每个优化器的 lr 都按"该优化器自己的合理范围"取——
# SGD/Momentum 偏大（0.1）；Adam/AdamW 因为有 √v̂ 归一化要小一档（1e-2）
results = {
    'SGD lr=0.1':                    train(torch.optim.SGD,   {'lr': 0.1}),
    'SGD+Momentum lr=0.1, μ=0.9':    train(torch.optim.SGD,   {'lr': 0.1, 'momentum': 0.9}),
    'Adam lr=1e-2':                  train(torch.optim.Adam,  {'lr': 1e-2}),
    # weight_decay=1e-2 是 AdamW 的标志性用法——L2 正则项被解耦到 update 末尾
    'AdamW lr=1e-2, wd=1e-2':        train(torch.optim.AdamW, {'lr': 1e-2, 'weight_decay': 1e-2}),
}

plt.figure(figsize=(8, 4))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Training curves of four optimizers on make_moons')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print('观察（玩具数据上不一定碾压，但趋势成立）：')
print('  SGD —— 收敛最慢；非凸地形里是最朴素的基线')
print('  +Momentum —— 比纯 SGD 显著快')
print('  Adam —— 自适应步长更稳健；这个 toy 数据 + 当前 lr 下，前期实际略慢于 Momentum，')
print('         因为单步被 √v̂ 归一到 ≈ lr=1e-2，Momentum 在 lr=0.1 下累积速度后单步反而更大')
print('  AdamW —— 与 Adam 相近，但 weight decay 是 LLM 长训练时稳定的关键')


In [ ]:
# ============================================================
# Cell 7: 真实训练循环里 warmup + cosine 怎么写
# ============================================================
# 这是把 P02 的训练循环骨架与 P04 的调度拼起来的"成品"模板。
# 后续读 LLM 训练脚本时，下面这套骨架几乎一字不差地能套上。
# 唯一会变的通常是：数据 / 模型 / batch loop（dataloader 替代这里的 full-batch X、y）。
import math
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import make_moons

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=0)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1))
# AdamW 是 LLM 预训练 / SFT 的事实标准——这里直接演示"生产配置"长什么样
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-2,                # peak lr（实际 LLM 预训练里量级是 1e-4 ~ 6e-4）
    betas=(0.9, 0.95),      # β2 比 0.999 略小，对早期更敏感（LLM 常见配置）
    weight_decay=1e-2,      # 把参数往 0 拉的强度；LLM 预训练常用 0.1
)

# warmup + cosine schedule
T_w = 50                    # warmup step 数（实际工程里 LLM 常取总步数的 1%~10%）
T   = 500                   # 总 step 数（也是这个例子的 epoch 数，因为是 full-batch）
eta_min_ratio = 0.1         # cosine 退火下界 = 10% peak

def lr_lambda(step):
    # 与 cell 5 完全相同的 warmup + cosine 形状，注释见 cell 5
    if step < T_w:
        return step / T_w                         # linear warmup
    progress = (step - T_w) / (T - T_w)
    return eta_min_ratio + 0.5 * (1 - eta_min_ratio) * (1 + math.cos(math.pi * progress))

# LambdaLR 是最通用的调度器：把 step → 倍率的任意函数喂给它都行
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

losses, lrs = [], []
for step in range(T):
    logits = model(X).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logits, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()                              # ① 先 optimizer.step
    scheduler.step()                              # ② 再 scheduler.step（绝不能反过来）

    losses.append(loss.item())
    # param_groups 是优化器的"参数组列表"；最常见情况只有一组，所以取 [0]
    # 取 'lr' 就是当前这一步实际生效的 lr——日志 / wandb 里就是这么打的
    lrs.append(optimizer.param_groups[0]['lr'])   # 当前 lr 直接从 param_groups 读

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
# 左图：lr vs step——能直接看出 warmup + cosine 形状是否对
axes[0].plot(lrs); axes[0].set_xlabel('step'); axes[0].set_ylabel('lr')
axes[0].set_title('Actual lr vs step (warmup + cosine)')
axes[0].axvline(T_w, color='gray', linestyle='--')
# 右图：loss vs step——配合左图看"lr 变化对 loss 的影响"
axes[1].plot(losses); axes[1].set_xlabel('step'); axes[1].set_ylabel('loss')
axes[1].set_title('Training loss curve')
plt.tight_layout()
plt.show()

print('要点回顾：')
print('  - optimizer.zero_grad() / loss.backward() / optimizer.step() 三件套不变')
print('  - scheduler.step() 在 optimizer.step() 之后调用')
print('  - 当前 lr 始终能从 optimizer.param_groups[0]["lr"] 读到，方便 logging')
